In [ ]:
#
# ⚡ UNIVERSAL FIRST CELL - Run this FIRST in every notebook!
# Compatible with: Google Colab, GitHub Codespaces, Local
#

import os
import subprocess
import sys

# Detect environment
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)

print(f"🚀 Environment: {'Colab' if IS_COLAB else 'Codespaces' if IS_CODESPACES else 'Local'}")

# Install required packages
!pip install -q yfinance statsmodels

# Ensure we are in the root directory for relative paths
while not os.path.exists('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print(f"📁 Working directory set to: {os.getcwd()}")

print("✅ Environment ready!")

# 16 - Stock Price Event Study

**Goal**: Measure stock market reaction to AI data center buildout announcements using event study methodology.

**Data**: `data/processed/buildout_promises_real.csv` — 5,295 buildout promise events from 2020–2026

**Methodology**: Market model (S&P 500 proxy via SPY) with estimation window [-120, -21] and event window [-20, +60] trading days around each announcement.

**Key Outputs**: Cumulative Abnormal Returns (CAR), significance tests, subsample analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Statsmodels for OLS market model
import statsmodels.api as sm

# yfinance for stock data
import yfinance as yf

# Scipy for significance tests
from scipy import stats as scipy_stats

print("✅ Libraries imported")

In [ ]:
# Load events
df = pd.read_csv('data/processed/buildout_promises_real.csv')
print(f"📊 Loaded {len(df)} events with {len(df.columns)} columns")

# Parse dates
df['event_date'] = pd.to_datetime(df['date'].astype(str), format='%Y%m%d%H%M%S', errors='coerce')
df['event_date_only'] = df['event_date'].dt.date

print(f"📅 Date range: {df['event_date'].min().date()} to {df['event_date'].max().date()}")
print(f"🏢 Unique companies: {df['company'].nunique()}")
print(f"📈 Total events: {len(df)}")

## Step 1: Event List Preparation

Map company names to stock tickers, filter to publicly traded companies with valid dates.
Crusoe is the only company without a public ticker (21 events dropped).

In [ ]:
# Company to ticker mapping
COMPANY_TICKER = {
    'Microsoft': 'MSFT', 'Microsoft Corp': 'MSFT',
    'Google': 'GOOGL', 'Alphabet': 'GOOGL',
    'Amazon': 'AMZN', 'Amazon Web Services': 'AMZN',
    'Facebook': 'META', 'Meta': 'META',
    'NVIDIA': 'NVDA', 'Oracle': 'ORCL',
    'Apple': 'AAPL', 'Digital Realty': 'DLR',
    'Equinix': 'EQIX', 'American Tower': 'AMT',
    'Prologis': 'PLD', 'First Industrial': 'FR',
}

df['ticker'] = df['company'].map(COMPANY_TICKER)

# Filter to events with valid ticker and date
events = df[df['ticker'].notna() & df['event_date'].notna()].copy()
print(f"✅ Events with valid ticker & date: {len(events)}")
print(f"   Dropped {len(df) - len(events)} events (Crusoe = not public)")
print()
print("Events per ticker:")
print(events['ticker'].value_counts().to_string())
print()
print("Confidence distribution:")
print(events['confidence'].value_counts().to_string())

## Step 2: Download Stock Data

Download daily OHLCV for each unique ticker plus SPY (S&P 500 market proxy) from Yahoo Finance.
Date range: June 2019 to June 2026 (includes padding for estimation windows).

In [ ]:
# Unique tickers
unique_tickers = sorted(events['ticker'].unique())
tickers_to_download = unique_tickers + ['SPY']
print(f"Downloading {len(tickers_to_download)} tickers: {', '.join(tickers_to_download)}")

# Date range with padding for estimation windows
start_date = '2019-06-01'  # ~6 months before first event for estimation
end_date = '2026-06-01'

# Download price data
print("Downloading from Yahoo Finance...")
data = yf.download(tickers_to_download, start=start_date, end=end_date, progress=False)

# Extract prices (handle different yfinance column structures)
if isinstance(data.columns, pd.MultiIndex):
    level_0 = data.columns.get_level_values(0)
    if 'Adj Close' in level_0:
        prices = data['Adj Close'].copy()
    else:
        prices = data['Close'].copy()
else:
    prices = data.copy()

# Check that all tickers downloaded
missing = [t for t in tickers_to_download if t not in prices.columns]
if missing:
    print(f"⚠️ Missing tickers: {missing}")
    tickers_to_download = [t for t in tickers_to_download if t not in missing]
    prices = prices[tickers_to_download]

print(f"✅ Downloaded {len(prices)} trading days")
print(f"   Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"   Tickers: {list(prices.columns)}")

# Compute daily returns
returns = prices.pct_change().dropna()
print(f"✅ Returns computed: {len(returns)} days")

# Market returns (SPY) and stock returns
market_returns = returns['SPY'].copy()
stock_tickers = [t for t in unique_tickers if t in returns.columns]
stock_returns = returns[stock_tickers]
print(f"   Stock tickers with data: {len(stock_tickers)}")

## Step 3: Market Model Estimation & CAR Computation

For each unique (ticker, date) pair:
1. Estimation window: [-120, -21] trading days before event — fit OLS: $R_{it} = \alpha_i + \beta_i R_{mt} + \varepsilon_{it}$
2. Event window: [-20, +60] trading days around event
3. Abnormal Return (AR): actual return minus predicted normal return
4. Cumulative Abnormal Return (CAR): sum of ARs over the event window

Report CAR for multiple windows: [-1, +1], [-5, +5], [-20, +60]

In [ ]:
# Deduplicate events by (ticker, date_only) to avoid same-day clustering
unique_events = events[['ticker', 'event_date']].drop_duplicates().reset_index(drop=True)
print(f"Unique (ticker, date) pairs to analyze: {len(unique_events)}")

def compute_event_car(event_date, ticker, stock_rets_df, market_rets_srs,
                      est_window=120, gap=21, ev_before=20, ev_after=60):
    """
    Compute CAR for a single event using the market model.
    
    Estimation window: [-est_window, -gap] trading days before event
    Event window: [-ev_before, +ev_after] trading days around event
    
    Returns dict with results or None if insufficient data.
    """
    # Check ticker exists
    if ticker not in stock_rets_df.columns:
        return None
    
    t_rets = stock_rets_df[ticker].dropna()
    
    # Convert event_date to Timestamp and find index in returns
    event_dt = pd.Timestamp(event_date)
    all_dates = t_rets.index
    
    # Find first trading day on or after event date
    mask = all_dates >= event_dt
    if not mask.any():
        return None
    event_idx = mask.argmax()
    
    if event_idx >= len(all_dates):
        return None
    
    actual_event_date = all_dates[event_idx]
    
    # Check bounds for estimation and event windows
    est_start = event_idx - est_window
    est_end = event_idx - gap  # exclusive end
    ev_start = event_idx - ev_before
    ev_end = event_idx + ev_after + 1  # inclusive -> exclusive
    
    if est_start < 0 or est_end <= est_start or ev_start < 0 or ev_end > len(all_dates):
        return None
    
    # Estimation window returns
    est_stock = t_rets.iloc[est_start:est_end]
    est_market = market_rets_srs.loc[est_stock.index]
    
    # Event window returns
    ev_stock = t_rets.iloc[ev_start:ev_end]
    ev_market = market_rets_srs.loc[ev_stock.index]
    
    # OLS market model: R_it = alpha + beta * R_mt + eps
    X_est = sm.add_constant(est_market.values)
    try:
        model = sm.OLS(est_stock.values, X_est).fit()
    except Exception:
        return None
    
    alpha = model.params[0]
    beta = model.params[1]
    
    # Predict normal returns in event window
    X_ev = sm.add_constant(ev_market.values)
    predicted = model.predict(X_ev)
    
    # Abnormal returns
    ar = ev_stock.values - predicted
    
    # Cumulative abnormal returns
    car_full = np.cumsum(ar)
    
    # CAR for specific windows
    # Event time index: -ev_before ... ev_after
    ev_time = np.arange(-ev_before, ev_after + 1)
    
    # Indices for specific windows
    def window_car(start_idx, end_idx):
        """Sum of AR in [start_idx, end_idx] relative to event time."""
        s = np.where(ev_time == start_idx)[0]
        e = np.where(ev_time == end_idx)[0]
        if len(s) == 0 or len(e) == 0:
            return np.nan
        return np.sum(ar[s[0]:e[0]+1])
    
    car_short = window_car(-1, 1)    # [-1, +1]
    car_medium = window_car(-5, 5)   # [-5, +5]
    car_long = window_car(-20, 60)   # [-20, +60]
    
    return {
        'ticker': ticker,
        'event_date': actual_event_date,
        'alpha': alpha,
        'beta': beta,
        'r_squared': model.rsquared,
        'n_est': len(est_stock),
        'ar_series': ar,
        'car_series': car_full,
        'event_time': ev_time,
        'car_m1p1': car_short,
        'car_m5p5': car_medium,
        'car_m20p60': car_long,
    }

# Compute CAR for each unique event
print(f"Computing market model for {len(unique_events)} events...")
all_results = []
failed = 0
for i, row in unique_events.iterrows():
    result = compute_event_car(row['event_date'], row['ticker'],
                               stock_returns, market_returns)
    if result is not None:
        all_results.append(result)
    else:
        failed += 1
    
    if (i + 1) % 500 == 0:
        print(f"  Progress: {i+1}/{len(unique_events)} events processed")

print(f"✅ Computed CAR for {len(all_results)} events")
print(f"⚠️ Failed (insufficient data): {failed} events")

# Build results DataFrame
car_df = pd.DataFrame({
    'ticker': [r['ticker'] for r in all_results],
    'event_date': [r['event_date'] for r in all_results],
    'alpha': [r['alpha'] for r in all_results],
    'beta': [r['beta'] for r in all_results],
    'r_squared': [r['r_squared'] for r in all_results],
    'n_est': [r['n_est'] for r in all_results],
    'car_m1p1': [r['car_m1p1'] for r in all_results],
    'car_m5p5': [r['car_m5p5'] for r in all_results],
    'car_m20p60': [r['car_m20p60'] for r in all_results],
})

print(f"\n📊 Results shape: {car_df.shape}")
print("\nMarket model diagnostics:")
print(f"  Mean alpha: {car_df['alpha'].mean():.6f}")
print(f"  Mean beta: {car_df['beta'].mean():.4f}")
print(f"  Mean R-squared: {car_df['r_squared']:.4f}")

## Step 4: Significance Testing

Test whether mean CAR is significantly different from zero using a cross-sectional t-test:

$$t = \frac{\overline{CAR}}{\sigma_{CAR} / \sqrt{N}}$$

where $\overline{CAR}$ is the mean cumulative abnormal return and $\sigma_{CAR}$ is the cross-sectional standard deviation.

In [ ]:
def significance_test(car_values, label=''):
    """Run cross-sectional t-test on CAR values."""
    car_values = car_values.dropna()
    n = len(car_values)
    mean_car = car_values.mean()
    median_car = car_values.median()
    std_car = car_values.std(ddof=1)
    se_car = std_car / np.sqrt(n)
    t_stat = mean_car / se_car
    p_value = 2 * (1 - scipy_stats.t.cdf(abs(t_stat), df=n-1))
    pct_positive = (car_values > 0).mean() * 100
    
    return {
        'window': label,
        'n': n,
        'mean_car': mean_car,
        'median_car': median_car,
        'std_car': std_car,
        'se_car': se_car,
        't_stat': t_stat,
        'p_value': p_value,
        'pct_positive': pct_positive,
        'significant_5pct': p_value < 0.05,
        'significant_1pct': p_value < 0.01,
    }

print("=" * 60)
print("   SIGNIFICANCE TESTS - Full Sample")
print("=" * 60)

results_list = []
for window, col in [('[-1, +1]', 'car_m1p1'), ('[-5, +5]', 'car_m5p5'), ('[-20, +60]', 'car_m20p60')]:
    res = significance_test(car_df[col], window)
    results_list.append(res)
    sig_mark = '***' if res['p_value'] < 0.01 else '**' if res['p_value'] < 0.05 else '*' if res['p_value'] < 0.1 else ''
    print(f"\n  Window {window}:")
    print(f"    Mean CAR: {res['mean_car']*100:+.4f}%  (t = {res['t_stat']:.3f}{sig_mark})")
    print(f"    Median CAR: {res['median_car']*100:+.4f}%")
    print(f"    Std Dev: {res['std_car']*100:.4f}%")
    print(f"    p-value: {res['p_value']:.6f}")
    print(f"    % Positive: {res['pct_positive']:.1f}%")
    print(f"    N: {res['n']}")

sig_df = pd.DataFrame(results_list)
print("\n" + "=" * 60)
print("Significance legend: *** p<0.01, ** p<0.05, * p<0.1")

## Step 5: Subsample Analysis

Split events by company, confidence tier, and announcement year to identify heterogeneous effects.

In [ ]:
# Merge company info, confidence, and year back into car_df
# Build lookup for first event per (ticker, date_only)
events['date_only'] = events['event_date_only']
lookup = events[['ticker', 'event_date', 'company', 'confidence']].drop_duplicates(
    subset=['ticker', 'event_date']
).set_index(['ticker', 'event_date'])

car_df['company'] = car_df.apply(
    lambda r: lookup.loc[(r['ticker'], r['event_date']), 'company']
    if (r['ticker'], r['event_date']) in lookup.index else '', axis=1
)
car_df['confidence'] = car_df.apply(
    lambda r: lookup.loc[(r['ticker'], r['event_date']), 'confidence']
    if (r['ticker'], r['event_date']) in lookup.index else '', axis=1
)
car_df['year'] = pd.DatetimeIndex(car_df['event_date']).year

print(f"Merged company/confidence/year info into CAR results.")

# --- By Company (top 10 by event count) ---
print("\n" + "=" * 60)
print("   SUBSAMPLE: By Company (CAR[-5,+5] window)")
print("=" * 60)

company_stats = car_df.groupby('company').agg(
    n=('car_m5p5', 'count'),
    mean_car=('car_m5p5', 'mean'),
    median_car=('car_m5p5', 'median'),
    std_car=('car_m5p5', 'std'),
    pct_positive=('car_m5p5', lambda x: (x > 0).mean() * 100)
).sort_values('n', ascending=False)

# Add t-stat and p-value
def comp_t_test(group):
    vals = group.dropna()
    n = len(vals)
    if n < 3:
        return (np.nan, np.nan, np.nan)
    se = vals.std(ddof=1) / np.sqrt(n)
    t = vals.mean() / se
    p = 2 * (1 - scipy_stats.t.cdf(abs(t), df=n-1))
    return (t, p, n)

t_stats, p_vals, ns = zip(*car_df.groupby('company')['car_m5p5'].apply(comp_t_test))
company_stats['t_stat'] = t_stats
company_stats['p_value'] = p_vals
company_stats['significant'] = company_stats['p_value'] < 0.05

print(company_stats.to_string(float_format=lambda x: f'{x:.6f}' if abs(x) < 0.01 else f'{x:.4f}'))

# --- By Confidence ---
print("\n" + "=" * 60)
print("   SUBSAMPLE: By Confidence Tier")
print("=" * 60)

conf_stats = car_df.groupby('confidence').agg(
    n=('car_m5p5', 'count'),
    mean_car=('car_m5p5', 'mean'),
    median_car=('car_m5p5', 'median'),
    std_car=('car_m5p5', 'std'),
    pct_positive=('car_m5p5', lambda x: (x > 0).mean() * 100)
).sort_index()

t_stats_c, p_vals_c, _ = zip(*car_df.groupby('confidence')['car_m5p5'].apply(comp_t_test))
conf_stats['t_stat'] = t_stats_c
conf_stats['p_value'] = p_vals_c
conf_stats['significant'] = conf_stats['p_value'] < 0.05

print(conf_stats.to_string(float_format=lambda x: f'{x:.6f}' if abs(x) < 0.01 else f'{x:.4f}'))

# --- By Year ---
print("\n" + "=" * 60)
print("   SUBSAMPLE: By Year")
print("=" * 60)

year_stats = car_df.groupby('year').agg(
    n=('car_m5p5', 'count'),
    mean_car=('car_m5p5', 'mean'),
    median_car=('car_m5p5', 'median'),
    std_car=('car_m5p5', 'std'),
    pct_positive=('car_m5p5', lambda x: (x > 0).mean() * 100)
)

t_stats_y, p_vals_y, _ = zip(*car_df.groupby('year')['car_m5p5'].apply(comp_t_test))
year_stats['t_stat'] = t_stats_y
year_stats['p_value'] = p_vals_y
year_stats['significant'] = year_stats['p_value'] < 0.05

print(year_stats.to_string(float_format=lambda x: f'{x:.6f}' if abs(x) < 0.01 else f'{x:.4f}'))

## Step 6: Visualizations

Three figures:
1. **CAR Event-Time Trend** — mean CAR across [-20, +60] with 95% confidence bands
2. **CAR Distribution** — histogram of individual event CARs
3. **Subsample Bar Chart** — mean CAR by company, confidence, and year with 95% CI

In [ ]:
# Build aggregate CAR time series across all events
# Align each event's CAR series by event time and compute mean/CI

# All results have the same event_time [-20, ..., 60] = 81 trading days
n_events = len(all_results)
n_days = len(all_results[0]['event_time'])

# Stack all CAR series into a matrix
car_matrix = np.zeros((n_events, n_days))
for i, r in enumerate(all_results):
    car_matrix[i, :] = r['car_series']

# Compute mean and confidence bands
mean_car_by_day = car_matrix.mean(axis=0)
se_car_by_day = car_matrix.std(axis=0, ddof=1) / np.sqrt(n_events)
ci_lower = mean_car_by_day - 1.96 * se_car_by_day
ci_upper = mean_car_by_day + 1.96 * se_car_by_day

event_time = all_results[0]['event_time']

# ---- FIGURE 1: CAR Event-Time Trend ----
fig1, ax1 = plt.subplots(figsize=(12, 6))

ax1.fill_between(event_time, ci_lower * 100, ci_upper * 100, alpha=0.2, color='steelblue', 
                 label='95% CI')
ax1.plot(event_time, mean_car_by_day * 100, color='steelblue', linewidth=2, label='Mean CAR (%)')
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.7)
ax1.axvline(x=0, color='red', linestyle=':', alpha=0.5, label='Event day')
ax1.set_xlabel('Trading Days Relative to Event')
ax1.set_ylabel('Cumulative Abnormal Return (%)')
ax1.set_title('Mean Cumulative Abnormal Return Around Buildout Announcements')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

plt.tight_layout()
fig1.savefig('reports/figures/fig12_car_trend.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved fig12_car_trend.png to reports/figures/")

# ---- FIGURE 2: CAR Distribution ([-5, +5] window) ----
fig2, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
cars = car_df['car_m5p5'].dropna() * 100
axes[0].hist(cars, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('CAR[-5, +5] (%)')
axes[0].set_ylabel('Number of Events')
axes[0].set_title(f'Distribution of Event CARs (n={len(cars)})')
axes[0].grid(True, alpha=0.3)

# Q-Q plot style: sorted CAR values
sorted_cars = np.sort(cars)
axes[1].plot(sorted_cars, np.arange(len(sorted_cars)) / len(sorted_cars), 
            color='steelblue', linewidth=2)
axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.7)
axes[1].set_xlabel('CAR[-5, +5] (%)')
axes[1].set_ylabel('Cumulative Probability')
axes[1].set_title('CDF of Event CARs')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig2.savefig('reports/figures/fig13_car_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved fig13_car_distribution.png to reports/figures/")

In [ ]:
# ---- FIGURE 3: Subsample Bar Charts ----
fig3, axes = plt.subplots(1, 3, figsize=(18, 6))

colors = {'high': '#2ecc71', 'medium': '#f39c12', 'low': '#e74c3c'}

# 3a: By Company (top 8)
top8 = company_stats.head(8)
top8 = top8.sort_values('mean_car')
bars1 = axes[0].barh(range(len(top8)), top8['mean_car'] * 100, 
                     xerr=1.96 * top8['std_car'] / np.sqrt(top8['n']) * 100,
                     capsize=3, color='steelblue', alpha=0.8)
axes[0].set_yticks(range(len(top8)))
axes[0].set_yticklabels(top8.index)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.7)
axes[0].set_xlabel('Mean CAR[-5,+5] (%)')
axes[0].set_title('Mean CAR by Company')
axes[0].grid(True, alpha=0.3, axis='x')

# 3b: By Confidence
conf_sorted = conf_stats.sort_values('mean_car')
conf_colors = [colors.get(c, 'steelblue') for c in conf_sorted.index]
bars2 = axes[1].barh(range(len(conf_sorted)), conf_sorted['mean_car'] * 100,
                     xerr=1.96 * conf_sorted['std_car'] / np.sqrt(conf_sorted['n']) * 100,
                     capsize=3, color=conf_colors, alpha=0.8)
axes[1].set_yticks(range(len(conf_sorted)))
axes[1].set_yticklabels([f"{c} (n={conf_sorted.loc[c,'n']})" for c in conf_sorted.index])
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Mean CAR[-5,+5] (%)')
axes[1].set_title('Mean CAR by Confidence Tier')
axes[1].grid(True, alpha=0.3, axis='x')

# 3c: By Year
year_sorted = year_stats.sort_index()
bars3 = axes[2].bar(range(len(year_sorted)), year_sorted['mean_car'] * 100,
                    yerr=1.96 * year_sorted['std_car'] / np.sqrt(year_sorted['n']) * 100,
                    capsize=3, color='steelblue', alpha=0.8)
axes[2].axhline(y=0, color='gray', linestyle='--', alpha=0.7)
axes[2].set_xticks(range(len(year_sorted)))
axes[2].set_xticklabels([f"{y}\n(n={year_sorted.loc[y,'n']})" for y in year_sorted.index], fontsize=9)
axes[2].set_ylabel('Mean CAR[-5,+5] (%)')
axes[2].set_title('Mean CAR by Announcement Year')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
fig3.savefig('reports/figures/fig14_car_subsamples.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved fig14_car_subsamples.png to reports/figures/")

## Step 7: Summary Statistics & LaTeX Output

Print key results as LaTeX tables for inclusion in the report.

In [ ]:
print("=" * 60)
print("   SUMMARY STATISTICS")
print("=" * 60)

# Overall summary
summary = pd.DataFrame({
    'Window': ['[-1, +1]', '[-5, +5]', '[-20, +60]'],
    'N': [sig_df.loc[sig_df['window'] == w, 'n'].values[0] for w in ['[-1, +1]', '[-5, +5]', '[-20, +60]']],
    'Mean CAR (%)': [sig_df.loc[sig_df['window'] == w, 'mean_car'].values[0] * 100 for w in ['[-1, +1]', '[-5, +5]', '[-20, +60]']],
    'Median CAR (%)': [sig_df.loc[sig_df['window'] == w, 'median_car'].values[0] * 100 for w in ['[-1, +1]', '[-5, +5]', '[-20, +60]']],
    'Std CAR (%)': [sig_df.loc[sig_df['window'] == w, 'std_car'].values[0] * 100 for w in ['[-1, +1]', '[-5, +5]', '[-20, +60]']],
    't-stat': [sig_df.loc[sig_df['window'] == w, 't_stat'].values[0] for w in ['[-1, +1]', '[-5, +5]', '[-20, +60]']],
    'p-value': [sig_df.loc[sig_df['window'] == w, 'p_value'].values[0] for w in ['[-1, +1]', '[-5, +5]', '[-20, +60]']],
    '% Positive': [sig_df.loc[sig_df['window'] == w, 'pct_positive'].values[0] for w in ['[-1, +1]', '[-5, +5]', '[-20, +60]']],
})

print()
print(summary.to_string(index=False))

print("\n" + "=" * 60)
print("   LATEX TABLE: Main Results")
print("=" * 60)
print()

# Format for LaTeX
latex_summary = summary.copy()
for col in ['Mean CAR (%)', 'Median CAR (%)', 'Std CAR (%)']:
    latex_summary[col] = latex_summary[col].apply(lambda x: f'{x:+.4f}\\%')
latex_summary['t-stat'] = latex_summary['t-stat'].apply(lambda x: f'{x:.3f}')
latex_summary['p-value'] = latex_summary['p-value'].apply(lambda x: f'{x:.4f}')
latex_summary['% Positive'] = latex_summary['% Positive'].apply(lambda x: f'{x:.1f}\\%')

print(latex_summary.to_latex(index=False, escape=False))

print("=" * 60)
print("   LATEX TABLE: Subsample by Company")
print("=" * 60)
print()

# Company subsample LaTeX
latex_company = company_stats.head(8).copy()
latex_company['mean_car'] = latex_company['mean_car'].apply(lambda x: f'{x*100:+.4f}\\%')
latex_company['median_car'] = latex_company['median_car'].apply(lambda x: f'{x*100:+.4f}\\%')
latex_company['std_car'] = latex_company['std_car'].apply(lambda x: f'{x*100:.4f}\\%')
latex_company['t_stat'] = latex_company['t_stat'].apply(lambda x: f'{x:.3f}' if pd.notna(x) else 'N/A')
latex_company['p_value'] = latex_company['p_value'].apply(lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A')
latex_company['pct_positive'] = latex_company['pct_positive'].apply(lambda x: f'{x:.1f}\\%')
latex_company['significant'] = latex_company['significant'].apply(lambda x: 'Yes' if x else 'No')

print(latex_company.to_latex(index=True, escape=False))

print("=" * 60)
print("   LATEX TABLE: Subsample by Confidence")
print("=" * 60)
print()

latex_conf = conf_stats.copy()
latex_conf['mean_car'] = latex_conf['mean_car'].apply(lambda x: f'{x*100:+.4f}\\%')
latex_conf['median_car'] = latex_conf['median_car'].apply(lambda x: f'{x*100:+.4f}\\%')
latex_conf['std_car'] = latex_conf['std_car'].apply(lambda x: f'{x*100:.4f}\\%')
latex_conf['t_stat'] = latex_conf['t_stat'].apply(lambda x: f'{x:.3f}')
latex_conf['p_value'] = latex_conf['p_value'].apply(lambda x: f'{x:.4f}')
latex_conf['pct_positive'] = latex_conf['pct_positive'].apply(lambda x: f'{x:.1f}\\%')
latex_conf['significant'] = latex_conf['significant'].apply(lambda x: 'Yes' if x else 'No')

print(latex_conf.to_latex(index=True, escape=False))

print("=" * 60)
print("   LATEX TABLE: Subsample by Year")
print("=" * 60)
print()

latex_year = year_stats.copy()
latex_year['mean_car'] = latex_year['mean_car'].apply(lambda x: f'{x*100:+.4f}\\%')
latex_year['median_car'] = latex_year['median_car'].apply(lambda x: f'{x*100:+.4f}\\%')
latex_year['std_car'] = latex_year['std_car'].apply(lambda x: f'{x*100:.4f}\\%')
latex_year['t_stat'] = latex_year['t_stat'].apply(lambda x: f'{x:.3f}')
latex_year['p_value'] = latex_year['p_value'].apply(lambda x: f'{x:.4f}')
latex_year['pct_positive'] = latex_year['pct_positive'].apply(lambda x: f'{x:.1f}\\%')
latex_year['significant'] = latex_year['significant'].apply(lambda x: 'Yes' if x else 'No')

print(latex_year.to_latex(index=True, escape=False))

## Summary

✅ **Event Study Complete**

- Market model estimated for each buildout announcement event
- CAR computed for [-1,+1], [-5,+5], and [-20,+60] trading day windows
- Cross-sectional t-tests assess statistical significance
- Subsample analysis reveals heterogeneity by company, confidence tier, and year
- Figures saved to `reports/figures/` (fig12, fig13, fig14)

**Caveats**:
- Event date precision limited to daily frequency (no intraday analysis)
- Market model assumes stable beta over estimation window
- Multiple events per company may introduce clustering (not corrected here)
- Crusoe (21 events) excluded as it is not publicly traded